In [ ]:
import queue
import sounddevice as sd
from vosk import Model, KaldiRecognizer
import json
import os

# Configuration
model_path = "vosk-model-small-id-0.22"
USE_DEMO_MODE = not os.path.exists(model_path)

if USE_DEMO_MODE:
    print("=" * 60)
    print("⚠️  VOSK MODEL NOT FOUND - RUNNING IN DEMO MODE")
    print("=" * 60)
    print("\n📥 To use real speech recognition:")
    print("  1. Download the model from: https://alphacephei.com/vosk/models")
    print("  2. Extract it to: ", os.path.abspath(model_path))
    print("  3. Re-run this cell\n")
    print("🎭 Demo mode will simulate speech recognition\n")
else:
    # Load Vosk model
    try:
        model = Model(model_path)
        recognizer = KaldiRecognizer(model, 16000)
        print("✅ Model loaded successfully!")
    except Exception as e:
        print(f"❌ Failed to load model: {e}")
        USE_DEMO_MODE = True
        print("Falling back to demo mode...\n")

# Audio queue
audio_q = queue.Queue()

def audio_callback(indata, frames, time, status):
    audio_q.put(bytes(indata))

if USE_DEMO_MODE:
    # Demo mode - simulates speech recognition
    print("🎤 Demo Mode: Listening... (enter text to simulate speech)")
    print("   Type 'exit' to stop\n")
    while True:
        try:
            text = input("🗣️ Simulated input: ").strip()
            if text.lower() == 'exit':
                break
            if text:
                print(f"✅ Recognized: {text}\n")
        except KeyboardInterrupt:
            break
        except EOFError:
            # Handle non-interactive environments
            print("Running in non-interactive mode. Cannot get input.")
            print("For real usage, download the model and re-run.")
            break
else:
    # Real speech recognition mode
    print("🎤 Listening... (say something)")
    with sd.RawInputStream(samplerate=16000, blocksize=8000, dtype='int16',
                           channels=1, callback=audio_callback):
        while True:
            try:
                data = audio_q.get()
                if recognizer.AcceptWaveform(data):
                    result = recognizer.Result()
                    text = json.loads(result)["text"]
                    if text:
                        print("🗣️ You said:", text)
            except KeyboardInterrupt:
                break

⚠️  VOSK MODEL NOT FOUND - RUNNING IN DEMO MODE

📥 To use real speech recognition:
  1. Download the model from: https://alphacephei.com/vosk/models
  2. Extract it to:  c:\Users\1just\Documents\ProjectIOT\project-iot\project-iot-main\backend\vosk-model-small-id-0.22
  3. Re-run this cell

🎭 Demo mode will simulate speech recognition

🎤 Demo Mode: Listening... (enter text to simulate speech)
   Type 'exit' to stop

✅ Recognized: enter



In [1]:
import numpy as np
import sounddevice as sd

print("🎤 Microphone Test")
print("=" * 60)

# List available audio devices
print("\n📊 Available Audio Devices:")
devices = sd.query_devices()
for i, device in enumerate(devices):
    print(f"{i}: {device['name']} (In: {device['max_input_channels']}, Out: {device['max_output_channels']})")

print("\n⏱️  Recording for 3 seconds...")
print("🔊 Speak into your microphone!\n")

# Record audio for 3 seconds
duration = 3  # seconds
samplerate = 16000  # Hz

try:
    # Record audio
    audio_data = sd.rec(int(samplerate * duration), samplerate=samplerate, 
                        channels=1, dtype='int16')
    sd.wait()  # Wait for recording to finish
    
    # Analyze the recording
    audio_level = np.abs(audio_data).mean()
    max_level = np.abs(audio_data).max()
    
    print("✅ Recording complete!\n")
    print(f"📈 Audio Statistics:")
    print(f"   Average Level: {audio_level:.2f}")
    print(f"   Max Level: {max_level:.2f}")
    print(f"   Audio Data Points: {len(audio_data)}")
    
    if audio_level < 100:
        print("\n⚠️  Low audio level detected!")
        print("   Try speaking louder or moving the mic closer")
    elif audio_level > 1000:
        print("\n✅ Good audio level detected!")
    
    # Play back the recording
    print("\n🔊 Playing back your recording...")
    sd.play(audio_data, samplerate)
    sd.wait()
    print("✅ Playback complete!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n💡 Troubleshooting tips:")
    print("   - Check if microphone is connected")
    print("   - Check Windows audio settings")
    print("   - Try a different audio device if available")

🎤 Microphone Test

📊 Available Audio Devices:
0: Microsoft Sound Mapper - Input (In: 2, Out: 0)
1: External Microphone (Realtek(R) (In: 2, Out: 0)
2: Microphone Array (Realtek(R) Au (In: 2, Out: 0)
3: Microphone (Steam Streaming Mic (In: 8, Out: 0)
4: Microsoft Sound Mapper - Output (In: 0, Out: 2)
5: Headphones (Realtek(R) Audio) (In: 0, Out: 2)
6: Speakers (Steam Streaming Micro (In: 0, Out: 8)
7: Speakers (Realtek(R) Audio) (In: 0, Out: 2)
8: Speakers (Steam Streaming Speak (In: 0, Out: 8)
9: Primary Sound Capture Driver (In: 2, Out: 0)
10: External Microphone (Realtek(R) Audio) (In: 2, Out: 0)
11: Microphone Array (Realtek(R) Audio) (In: 2, Out: 0)
12: Microphone (Steam Streaming Microphone) (In: 8, Out: 0)
13: Primary Sound Driver (In: 0, Out: 2)
14: Headphones (Realtek(R) Audio) (In: 0, Out: 2)
15: Speakers (Steam Streaming Microphone) (In: 0, Out: 8)
16: Speakers (Realtek(R) Audio) (In: 0, Out: 2)
17: Speakers (Steam Streaming Speakers) (In: 0, Out: 8)
18: Speakers (Steam Stream